## PURPOSE OF THIS NOTEBOOK:  
  
This notebook takes in a dataframe and transforms the data to be a format that Llama expects. We are expecing 3 columns:  
  
***Instruction***  
***Input***  
***Output***  
  
I would want to create 2 versions of this, One version that focuses on telling it to classify which ONET family it belongs too, and another one that will focus on the specific ONET, which will need to be formatted in a special manner (maybe another column for each ONET family where we can paste a formatted version that includes all the onet names).

In [65]:
import pandas as pd
import numpy as np
import csv

## Import the orignal data to create new Train/Test Data
  
To make creating the train/test quite easy, I can 

In [66]:
# Start by loading in the data. 
onet_df = pd.read_csv("../Data/Updated_ONET_Alt_Titles.csv")
base_onet_df = pd.read_csv("../Data/Occupation Data.csv")

# Filter out any columns we don't need. 
onet_filtered = onet_df[['O*NET-SOC Code', 'Reported Job Title']]

# Show the filtered data
print(onet_filtered.shape)
onet_filtered.head()

(44545, 2)


,O*NET-SOC Code,Reported Job Title
0,11-1011.00,Chief Diversity Officer (CDO)
1,11-1011.00,Chief Executive Officer (CEO)
2,11-1011.00,Chief Financial Officer (CFO)
3,11-1011.00,Chief Nursing Officer
4,11-1011.00,Chief Operating Officer (COO)


In [67]:
# remove the military onet codes. 
base_onet_df = base_onet_df[~base_onet_df['O*NET-SOC Code'].str[:2].isin(['55'])]
onet_filtered = onet_filtered[~onet_filtered['O*NET-SOC Code'].str[:2].isin(['55'])]

In [68]:
onet_filtered

,O*NET-SOC Code,Reported Job Title
0,11-1011.00,Chief Diversity Officer (CDO)
1,11-1011.00,Chief Executive Officer (CEO)
2,11-1011.00,Chief Financial Officer (CFO)
3,11-1011.00,Chief Nursing Officer
4,11-1011.00,Chief Operating Officer (COO)
...,...,...
43946,53-7199.00,Water Hauler
43947,53-7199.00,Windlasser
43948,53-7199.00,Wire Wheeler
43949,53-7199.00,Yarn Man


In [69]:
base_onet_df.head()

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."


In [70]:
onet_filtered.head()

,O*NET-SOC Code,Reported Job Title
0,11-1011.00,Chief Diversity Officer (CDO)
1,11-1011.00,Chief Executive Officer (CEO)
2,11-1011.00,Chief Financial Officer (CFO)
3,11-1011.00,Chief Nursing Officer
4,11-1011.00,Chief Operating Officer (COO)


In [71]:
# Create the Family group Column
base_onet_df['O*NET-SOC Group'] = base_onet_df['O*NET-SOC Code'].str[:2]
onet_filtered['O*NET-SOC Group'] = onet_filtered['O*NET-SOC Code'].str[:2]

# Filter out the SOC codes 
base_onet_df = base_onet_df[['Title', 'O*NET-SOC Group']]
onet_filtered = onet_filtered[['O*NET-SOC Group', 'Reported Job Title']]

# rename to match what is expected
base_onet_df.rename(columns={'Title':'input', 'O*NET-SOC Group':'output'}, inplace=True)
onet_filtered.rename(columns={'Reported Job Title':'input', 'O*NET-SOC Group':'output'}, inplace=True)

/tmp/ipykernel_6986/1697478149.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  onet_filtered.rename(columns={'Reported Job Title':'input', 'O*NET-SOC Group':'output'}, inplace=True)


In [72]:
# Merge the two dataframes 
print(f'base_onet_df size: {base_onet_df.shape}')
print(f'filtered onet size: {onet_filtered.shape}')
full_onet_df = pd.concat([base_onet_df, onet_filtered], ignore_index=True)
full_onet_df.reset_index(inplace=True, drop=True)
print(f'merged dataset size: {full_onet_df.shape}')

base_onet_df size: (997, 2)
filtered onet size: (43951, 2)
merged dataset size: (44948, 2)


In [73]:
# add the instructions to the dataframe
full_onet_df['instruction'] = "Categorize the job title into one of the 22 job families:\n\n11\n13\n15\n17\n19\n21\n23\n25\n27\n29\n31\n33\n35\n37\n39\n41\n43\n45\n47\n49\n51\n53\n\n"

# reorder the columns 
col_list = ['instruction', 'input', 'output']
full_onet_df = full_onet_df[col_list]

# Check to make sure it is properlly inserted
full_onet_df.head()

,instruction,input,output
0,Categorize the job title into one of the 22 jo...,Chief Executives,11
1,Categorize the job title into one of the 22 jo...,Chief Sustainability Officers,11
2,Categorize the job title into one of the 22 jo...,General and Operations Managers,11
3,Categorize the job title into one of the 22 jo...,Legislators,11
4,Categorize the job title into one of the 22 jo...,Advertising and Promotions Managers,11


In [74]:
# Preform the train test split on the data. 
def tt_split(df):
    ''' 1. Define the final dataframes that we need     
        2. Create a for loop that will iterate through all of the rows and do a 50/20/30 split on the data. 
        3. Append the new data to the final dataframes
        4. Reset the index of the final dataframes 
        5. Export the final dataframes. 
        STRETCH GOAL: take this function and convert the dataframe equations to numpy array equations, will be much faster.  ''' 

    # Define the final dataframes 
    test_df = pd.DataFrame(columns=['instruction', 'input', 'output'])
    train_df = pd.DataFrame(columns=['instruction', 'input', 'output'])
    validation_df = pd.DataFrame(columns=['instruction', 'input', 'output'])

    # Grab all the columns apart from the final reported job title
    label_list = df.output.unique().tolist()
    
    for onet in label_list:
        filter_df = df.loc[df['output'] == onet]
        temp_train_df = filter_df.sample(frac=.5,random_state=150)

        temp_df = filter_df.drop(temp_train_df.index).reset_index(drop=True)
        temp_valid_df = temp_df.sample(frac=.4,random_state=150)
        temp_test_df = temp_df.drop(temp_valid_df.index).reset_index(drop=True)
        temp_train_df.reset_index(inplace=True, drop=True)
        temp_valid_df.reset_index(inplace=True, drop=True)

        # Append the new data to the final train/test dataframes 
        train_df = pd.concat([train_df, temp_train_df], ignore_index=True)
        test_df = pd.concat([test_df, temp_test_df], ignore_index=True)
        validation_df = pd.concat([validation_df, temp_valid_df], ignore_index=True)

    train_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)
    validation_df.reset_index(drop=True, inplace=True)
    

    return train_df, test_df, validation_df

In [75]:
train_df, test_df, validation_df = tt_split(full_onet_df)

In [76]:
# check to ensure the split was as intended:
print(train_df.shape)
train_df.head()

(22472, 3)


,instruction,input,output
0,Categorize the job title into one of the 22 jo...,Oil Well Drilling Manager,11
1,Categorize the job title into one of the 22 jo...,Craft Center Director,11
2,Categorize the job title into one of the 22 jo...,Recreational Resort Manager,11
3,Categorize the job title into one of the 22 jo...,Dairy Farm Manager,11
4,Categorize the job title into one of the 22 jo...,Human Resources Manager (HR Manager),11


In [77]:
print(test_df.shape)
test_df.head()

(13482, 3)


,instruction,input,output
0,Categorize the job title into one of the 22 jo...,Marketing Managers,11
1,Categorize the job title into one of the 22 jo...,Financial Managers,11
2,Categorize the job title into one of the 22 jo...,Treasurers and Controllers,11
3,Categorize the job title into one of the 22 jo...,Investment Fund Managers,11
4,Categorize the job title into one of the 22 jo...,Industrial Production Managers,11


In [78]:
print(validation_df.shape)
validation_df.head()

(8994, 3)


,instruction,input,output
0,Categorize the job title into one of the 22 jo...,Weatherization Operations Manager,11
1,Categorize the job title into one of the 22 jo...,Entertainment and Recreation Planning Manager,11
2,Categorize the job title into one of the 22 jo...,Sustainability Programs Director,11
3,Categorize the job title into one of the 22 jo...,Product Marketing Manager,11
4,Categorize the job title into one of the 22 jo...,Environmental Control Administrator,11


In [79]:
# Export 
train_df.to_csv('../Data/Training_Data.csv', index=False)
test_df.to_csv('../Data/TestingData.csv', index=False)
validation_df.to_csv('../Data/ValidationData.csv', index=False)